In [1]:
import google.cloud.aiplatform as aiplatform
import os
# Import the configuration classes specifically for enums if needed
from google.cloud.aiplatform.matching_engine import matching_engine_index_config

# --- Configuration ---
PROJECT_ID = "medical-insurance-457015"  # Replace with your project ID
REGION = "us-central1"            # Replace with your desired region (e.g., "us-central1")
BUCKET_NAME = "medical-insurance-vector-search"    # Replace with your Cloud Storage bucket name
FOLDER_NAME = "vector_search_staging/"    # Replace with the folder containing your JSONL files
INDEX_DISPLAY_NAME = "my-vector-search-index" # A display name for your index
EMBEDDING_DIMENSIONS = 384        # The dimension of your embedding vectors (based on your data)

# The Cloud Storage URI where your embedding data is located
GCS_INPUT_URI = f"gs://{BUCKET_NAME}/{FOLDER_NAME}"

# --- Initialize Vertex AI SDK ---
aiplatform.init(project=PROJECT_ID, location=REGION)

# --- Create the Index ---
print(f"Creating Vector Search index: {INDEX_DISPLAY_NAME} from {GCS_INPUT_URI}")

try:
    index = aiplatform.MatchingEngineIndex.create_tree_ah_index(
        display_name=INDEX_DISPLAY_NAME,
        contents_delta_uri=GCS_INPUT_URI,
        # Pass configuration parameters directly
        dimensions=EMBEDDING_DIMENSIONS,
        approximate_neighbors_count=150, # Adjust based on desired recall vs latency
        distance_measure_type=matching_engine_index_config.DistanceMeasureType.DOT_PRODUCT_DISTANCE, # Using the enum
        leaf_node_embedding_count=500, # Tree-AH specific parameter
        leaf_nodes_to_search_percent=7.0, # Tree-AH specific parameter
        description="My index for embeddings with metadata", # Optional description
    )

    print(f"Index creation initiated. Index resource name: {index.resource_name}")
    print("Index creation can take some time. You can monitor the progress in the Google Cloud Console.")

except Exception as e:
    print(f"An error occurred during index creation: {e}")

Creating Vector Search index: my-vector-search-index from gs://medical-insurance-vector-search/vector_search_staging/
Creating MatchingEngineIndex
Create MatchingEngineIndex backing LRO: projects/966745344354/locations/us-central1/indexes/8958998864002547712/operations/8223275604762427392
MatchingEngineIndex created. Resource name: projects/966745344354/locations/us-central1/indexes/8958998864002547712
To use this MatchingEngineIndex in another session:
index = aiplatform.MatchingEngineIndex('projects/966745344354/locations/us-central1/indexes/8958998864002547712')
Index creation initiated. Index resource name: projects/966745344354/locations/us-central1/indexes/8958998864002547712
Index creation can take some time. You can monitor the progress in the Google Cloud Console.


In [2]:
import google.cloud.aiplatform as aiplatform
import os

# --- Configuration ---
PROJECT_ID = "medical-insurance-457015"  # Replace with your project ID
REGION = "us-central1"            # Replace with your desired region (e.g., "us-central1")
INDEX_ENDPOINT_DISPLAY_NAME = "my-vector-search-endpoint" # A display name for your endpoint

# --- Initialize Vertex AI SDK ---
aiplatform.init(project=PROJECT_ID, location=REGION)

# --- Create the Index Endpoint ---
print(f"Creating Vertex AI Vector Search Index Endpoint: {INDEX_ENDPOINT_DISPLAY_NAME} in {REGION}")

try:
    # Use the correct class name and add the public_endpoint_enabled parameter
    index_endpoint = aiplatform.MatchingEngineIndexEndpoint.create(
        display_name=INDEX_ENDPOINT_DISPLAY_NAME,
        public_endpoint_enabled=True, # Explicitly enable a public endpoint
        # If you need a private endpoint with VPC network peering, uncomment and configure the network parameter INSTEAD:
        # network="projects/YOUR_PROJECT_NUMBER/global/networks/YOUR_VPC_NETWORK_NAME", # Replace YOUR_PROJECT_NUMBER and YOUR_VPC_NETWORK_NAME
        # Make sure the network is in the same region as the endpoint.
        # Or for Private Service Connect:
        # enable_private_service_connect=True # Uncomment and set to True for PSC
    )

    print(f"Index Endpoint creation initiated. Endpoint resource name: {index_endpoint.resource_name}")
    print("Index Endpoint creation can take some time. You can monitor the progress in the Google Cloud Console.")
    print(f"Once the endpoint is created and ready, you can deploy your index to it.")

except Exception as e:
    print(f"An error occurred during Index Endpoint creation: {e}")

Creating Vertex AI Vector Search Index Endpoint: my-vector-search-endpoint in us-central1
Creating MatchingEngineIndexEndpoint
Create MatchingEngineIndexEndpoint backing LRO: projects/966745344354/locations/us-central1/indexEndpoints/6149948965174378496/operations/3405268428406128640
MatchingEngineIndexEndpoint created. Resource name: projects/966745344354/locations/us-central1/indexEndpoints/6149948965174378496
To use this MatchingEngineIndexEndpoint in another session:
index_endpoint = aiplatform.MatchingEngineIndexEndpoint('projects/966745344354/locations/us-central1/indexEndpoints/6149948965174378496')
Index Endpoint creation initiated. Endpoint resource name: projects/966745344354/locations/us-central1/indexEndpoints/6149948965174378496
Index Endpoint creation can take some time. You can monitor the progress in the Google Cloud Console.
Once the endpoint is created and ready, you can deploy your index to it.


In [3]:
import google.cloud.aiplatform as aiplatform
import os

# --- Configuration ---
PROJECT_ID = "medical-insurance-457015"  # Replace with your project ID
REGION = "us-central1"            # Replace with your desired region (e.g., "us-central1")

# !!! IMPORTANT !!!
# Replace these with the exact resource names you obtained:
# Corrected Index Resource Name:
INDEX_RESOURCE_NAME = "projects/966745344354/locations/us-central1/indexes/8958998864002547712" # <-- Corrected Index resource name
INDEX_ENDPOINT_RESOURCE_NAME = "projects/966745344354/locations/us-central1/indexEndpoints/6149948965174378496" # <-- Your actual endpoint resource name

# Configuration for the deployed index on the endpoint
# !!! CHANGE THIS ID TO SOMETHING NEW AND UNIQUE IF you tried 'my_medical_index_v2' and got a conflict !!!
DEPLOYED_INDEX_ID = "my_medical_index_v2" # <-- Ensure this is unique if you've deployed before
DEPLOYED_INDEX_DISPLAY_NAME = "My Medical Insurance Index" # A display name for the deployed index

# Serving machine configuration (adjusted for shard size requirement)
MACHINE_TYPE = "e2-standard-16" # <-- Using e2-standard-16
MIN_REPLICA_COUNT = 1 # Minimum number of serving nodes
MAX_REPLICA_COUNT = 1 # Maximum number of serving nodes (can keep low for POC)

# --- Initialize Vertex AI SDK ---
aiplatform.init(project=PROJECT_ID, location=REGION)

# --- Get References to the Index and Index Endpoint ---
print(f"Attempting to reference Index: {INDEX_RESOURCE_NAME}")
print(f"Attempting to reference Index Endpoint: {INDEX_ENDPOINT_RESOURCE_NAME}")
try:
    # Get references to the existing index and endpoint using their resource names
    # Pass the resource name as a positional argument to the constructors
    index = aiplatform.MatchingEngineIndex(INDEX_RESOURCE_NAME)
    index_endpoint = aiplatform.MatchingEngineIndexEndpoint(INDEX_ENDPOINT_RESOURCE_NAME)
    print(f"Successfully referenced Index: {index.resource_name}")
    print(f"Successfully referenced Index Endpoint: {index_endpoint.resource_name}")

except Exception as e:
    print(f"Error referencing Index or Index Endpoint: {e}")
    print("Please ensure the resource names in the configuration are EXACTLY correct and the resources exist.")
    exit() # Exit if we can't reference the resources

# --- Deploy the Index to the Endpoint ---
print(f"\nDeploying Index '{index.resource_name}' to Endpoint '{index_endpoint.resource_name}' with machine type {MACHINE_TYPE}")

try:
    # The deploy_index method initiates the deployment.
    # The returned object is likely the updated endpoint or an LRO.
    deployed_index = index_endpoint.deploy_index(
        index=index,
        deployed_index_id=DEPLOYED_INDEX_ID, # Use the chosen unique ID
        display_name=DEPLOYED_INDEX_DISPLAY_NAME,
        machine_type=MACHINE_TYPE, # Using the e2-standard-16 machine type
        min_replica_count=MIN_REPLICA_COUNT, # Using minimum replica count
        max_replica_count=MAX_REPLICA_COUNT, # Using minimum replica count for max
        # Set to True if your endpoint was created with VPC network peering
        # enable_private_endpoints=True, # Uncomment and set to True if using a private endpoint
    )

    # Corrected print statements using the DEPLOYED_INDEX_ID variable
    print(f"Index deployment initiated.")
    print(f"Deployed Index ID (provided input): {DEPLOYED_INDEX_ID}") # Use the variable directly
    print(f"Deployed Index Display Name (provided input): {DEPLOYED_INDEX_DISPLAY_NAME}") # Use the variable directly
    print("Index deployment can take some time (typically 15-30 minutes or more depending on index size and machine type).")
    print("You can monitor the deployment progress in the Google Cloud Console on the Index Endpoint details page.")
    print(f"Once deployment is complete, the endpoint's public/private IP address will be available for querying.")
    print(f"You can also find the deployed index details by refreshing the Index Endpoint page in the console after deployment finishes.")


except Exception as e:
    print(f"An error occurred during index deployment: {e}")
    print("Please ensure the Index Endpoint is in the 'Ready' state and the machine type is compatible with the index shard size.")

Attempting to reference Index: projects/966745344354/locations/us-central1/indexes/8958998864002547712
Attempting to reference Index Endpoint: projects/966745344354/locations/us-central1/indexEndpoints/6149948965174378496
Successfully referenced Index: projects/966745344354/locations/us-central1/indexes/8958998864002547712
Successfully referenced Index Endpoint: projects/966745344354/locations/us-central1/indexEndpoints/6149948965174378496

Deploying Index 'projects/966745344354/locations/us-central1/indexes/8958998864002547712' to Endpoint 'projects/966745344354/locations/us-central1/indexEndpoints/6149948965174378496' with machine type e2-standard-16
Deploying index MatchingEngineIndexEndpoint index_endpoint: projects/966745344354/locations/us-central1/indexEndpoints/6149948965174378496
Deploy index MatchingEngineIndexEndpoint index_endpoint backing LRO: projects/966745344354/locations/us-central1/indexEndpoints/6149948965174378496/operations/5165049982801149952
MatchingEngineIndexEn

In [5]:
import json
from google.cloud import storage
import io

# --- Configuration ---
PROJECT_ID = "medical-insurance-457015"  # Replace with your project ID
BUCKET_NAME = "medical-insurance-vector-search" # Replace with your Cloud Storage bucket name
FILE_PATH_IN_BUCKET = "vector_search_staging/vertex_ready_vectors.jsonl" # Replace with the path to your file within the bucket
CHUNK_SIZE = 1024 # Read in 1KB chunks

# --- Initialize Google Cloud Storage Client ---
storage_client = storage.Client(project=PROJECT_ID)
bucket = storage_client.get_bucket(BUCKET_NAME)
blob = bucket.blob(FILE_PATH_IN_BUCKET)

first_line = ""
buffer = ""
start_byte = 0

try:
    # Read in chunks until a newline is found
    while True:
        chunk_bytes = blob.download_as_bytes(start=start_byte, end=start_byte + CHUNK_SIZE - 1)

        if not chunk_bytes:
            # End of file reached before finding a newline
            buffer += buffer # Add any remaining buffer to itself to ensure it's processed
            break

        chunk_text = chunk_bytes.decode('utf-8') # Decode the chunk

        buffer += chunk_text
        newline_index = buffer.find('\n')

        if newline_index != -1:
            first_line = buffer[:newline_index]
            break

        start_byte += CHUNK_SIZE # Move to the next chunk

    if not first_line:
         print(f"Could not read a complete first line from gs://{BUCKET_NAME}/{FILE_PATH_IN_BUCKET}. The file might be empty or the first line is extremely long.")
    else:
        # Parse the first line as a JSON object
        data = json.loads(first_line)

        # Check if the 'embedding' key exists and is a list
        if 'embedding' in data and isinstance(data['embedding'], list):
            embedding_dimension = len(data['embedding'])
            print(f"The dimension of embeddings in the file is: {embedding_dimension}")
        else:
            print(f"The first line of the file does not contain a valid 'embedding' list with an embedding.")

except Exception as e:
    print(f"An error occurred: {e}")

The dimension of embeddings in the file is: 384


In [7]:
pip show google.cloud.aiplatform

Name: google-cloud-aiplatform
Version: 1.90.0
Summary: Vertex AI API client library
Home-page: https://github.com/googleapis/python-aiplatform
Author: Google LLC
Author-email: googleapis-packages@google.com
License: Apache 2.0
Location: /opt/conda/lib/python3.10/site-packages
Requires: docstring-parser, google-api-core, google-auth, google-cloud-bigquery, google-cloud-resource-manager, google-cloud-storage, packaging, proto-plus, protobuf, pydantic, shapely, typing-extensions
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [16]:
pip show google-cloud-aiplatform


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Name: google-cloud-aiplatform
Version: 1.90.0
Summary: Vertex AI API client library
Home-page: https://github.com/googleapis/python-aiplatform
Author: Google LLC
Author-email: googleapis-packages@google.com
License: Apache 2.0
Location: /opt/conda/lib/python3.10/site-packages
Requires: docstring-parser, google-api-core, google-auth, google-cloud-bigquery, google-cloud-resource-manager, google-cloud-storage, packaging, proto-plus, protobuf, pydantic, shapely, typing-extensions
Required-by: 
Note: you may need to restart the kernel to use updated packages.
